# Initial 2023

In [1]:

# ============================================
# Spatio-temporal GP (validation) + PM2.5 full fit/imputation
# + County-week aggregation using joint covariance (MVE/GLS)
# ============================================
import time, warnings
import numpy as np
import pandas as pd
from itertools import combinations
from math import sqrt, log
from scipy.optimize import fmin_l_bfgs_b, brentq

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    Kernel, RBF, Matern, ExpSineSquared, WhiteKernel, ConstantKernel as C
)
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------
# Quiet noisy warnings
# -----------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------
# Global Config
# -----------------------------
PATH = "full_2023_weekly_averages.csv"
RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)

POLLUTANTS = ["PM2.5", "NO2", "CO"]
DEMEAN_BY_SITE = True
Y_TRANSFORM = {'PM2.5': 'yeojohnson', 'NO2': 'yeojohnson', 'CO': 'none'}

# Validation
MAX_TRAIN_PER = {'PM2.5': 1000, 'NO2': 1000, 'CO': 1000}
STRIDE_PER    = {'PM2.5': 1,    'NO2': 1,    'CO': 1}
RECENT_WEEKS  = 52
MAXITER_LBFGS = 20
RESTARTS_PER  = {'PM2.5': 1, 'NO2': 1, 'CO': 1}
DELTA_TARGET = {'NO2', 'CO'}
USE_LAG2_ONLY_FOR_DELTA = True

# Periodicity probe (used for bounds only)
PERIOD_WEEKS   = 52
ACF52_THRESHOLD = 0.2

# Variogram settings
MIN_PAIRS_PER_BIN    = 100
MAX_TEMP_LAG_WEEKS   = 52
SPATIAL_BIN_WIDTH_KM = 10.0
MAX_DISTANCE_KM      = 500.0
MAX_WEEKS_FOR_SPATIAL= 30
MAX_PAIRS_PER_WEEK   = 3000

# Full PM2.5 fit/impute
PM_TF = "yeojohnson"     # target transform for PM2.5 ('yeojohnson' or 'none')
MAX_TRAIN_ALL = 3000
RECENT_WEEKS_ALL = 52
STRIDE_ALL = 1

# County-week covariance aggregation
INCLUDE_OBS_NOISE = True   # True => keep obs noise in Σ; False => subtract WhiteKernel variance
RIDGE = 1e-8               # diag ridge for GLS/MVE stability

# -----------------------------
# Optimizer wrapper (capped L-BFGS-B)
# -----------------------------
def lbfgs_capped(obj_func, initial_theta, bounds):
    theta_opt, fmin, _ = fmin_l_bfgs_b(
        obj_func, initial_theta, bounds=bounds, maxiter=MAXITER_LBFGS
    )
    return theta_opt, fmin

# -----------------------------
# SliceKernel (feature selector)
# -----------------------------
class SliceKernel(Kernel):
    def __init__(self, base_kernel, active_dims):
        self.base_kernel = base_kernel
        self.active_dims = tuple(active_dims)
    def __call__(self, X, Y=None, eval_gradient=False):
        idx = np.asarray(self.active_dims)
        Xs = X[:, idx]
        Ys = None if Y is None else Y[:, idx]
        return self.base_kernel(Xs, Ys, eval_gradient=eval_gradient)
    def diag(self, X):
        idx = np.asarray(self.active_dims)
        return self.base_kernel.diag(X[:, idx])
    def is_stationary(self):
        return getattr(self.base_kernel, "is_stationary", lambda: False)()
    def get_params(self, deep=True):
        params = {"base_kernel": self.base_kernel, "active_dims": self.active_dims}
        if deep and hasattr(self.base_kernel, "get_params"):
            for k, v in self.base_kernel.get_params(deep=True).items():
                params[f"base_kernel__{k}"] = v
        return params
    def set_params(self, **params):
        base_params = {k.split("__",1)[1]: v for k,v in params.items() if k.startswith("base_kernel__")}
        if base_params and hasattr(self.base_kernel, "set_params"):
            self.base_kernel.set_params(**base_params)
        if "base_kernel" in params: self.base_kernel = params["base_kernel"]
        if "active_dims" in params: self.active_dims = tuple(params["active_dims"])
        return self
    @property
    def hyperparameters(self): return self.base_kernel.hyperparameters
    @property
    def theta(self): return self.base_kernel.theta
    @theta.setter
    def theta(self, t): self.base_kernel.theta = t
    @property
    def bounds(self): return self.base_kernel.bounds
    def clone_with_theta(self, theta): return SliceKernel(self.base_kernel.clone_with_theta(theta), self.active_dims)
    def __repr__(self): return f"SliceKernel(base_kernel={self.base_kernel!r}, active_dims={self.active_dims!r})"

# -----------------------------
# Utilities
# -----------------------------
def temporal_holdout_per_site(dfi, group_col='unique_id', date_col='ds', test_frac=0.2):
    dfi = dfi.sort_values([group_col, date_col]).reset_index(drop=True)
    is_test = np.zeros(len(dfi), dtype=bool)
    for _, idx in dfi.groupby(group_col).indices.items():
        k = max(1, int(np.ceil(test_frac * len(idx))))
        is_test[idx[-k:]] = True
    return dfi[~is_test].copy(), dfi[is_test].copy()

def thin_training(train_df, stride, max_total, recent_weeks, random_state=RANDOM_STATE,
                  min_per_site=5, frac_recent=0.7):
    rng = np.random.RandomState(random_state)
    if (max_total is None) and (stride == 1):
        return train_df

    parts_recent, parts_older = [], []
    for uid, g in train_df.groupby('unique_id', sort=False):
        g = g.sort_values('t_week', kind='mergesort')
        cutoff = g['t_week'].max() - recent_weeks
        gr = g[g['t_week'] >= cutoff]
        go = g[g['t_week'] <  cutoff]
        if stride > 1:
            if len(gr) > 0: gr = gr.iloc[::stride]
            if len(go) > 0: go = go.iloc[::stride]
        parts_recent.append(gr)
        parts_older.append(go)

    recent = pd.concat(parts_recent, ignore_index=False) if parts_recent else train_df.iloc[0:0]
    older  = pd.concat(parts_older,  ignore_index=False) if parts_older  else train_df.iloc[0:0]

    if max_total is None:
        return pd.concat([recent, older], ignore_index=True)

    site_counts = train_df['unique_id'].value_counts()
    quotas = (site_counts / site_counts.sum() * max_total).round().astype(int).clip(lower=min_per_site)

    chosen = []
    for uid, q in quotas.items():
        gr = recent[recent['unique_id'] == uid]
        go = older[older['unique_id'] == uid]
        q_recent = int(np.ceil(q * frac_recent))
        q_older  = q - q_recent
        pick_r = gr if len(gr) <= q_recent else gr.sample(q_recent, random_state=rng)
        pick_o = go if len(go) <= q_older  else go.sample(q_older,  random_state=rng)
        chosen.append(pd.concat([pick_r, pick_o], ignore_index=False))

    sub = pd.concat(chosen, ignore_index=True)
    if len(sub) > max_total:
        sub = sub.sample(max_total, random_state=rng)
    return sub

def nlpds(y_t_true, mu_t, sigma_t):
    sigma2 = np.maximum(sigma_t**2, 1e-12)
    return 0.5*np.log(2*np.pi*sigma2) + 0.5*((y_t_true - mu_t)**2)/sigma2

def find_slice_by_dims(k, dims):
    if isinstance(k, SliceKernel) and tuple(k.active_dims) == tuple(dims):
        return k
    for child in ("k1","k2"):
        if hasattr(k, child):
            ret = find_slice_by_dims(getattr(k, child), dims)
            if ret is not None: return ret
    return None

# -----------------------------
# Load data & features
# -----------------------------
df_raw = pd.read_csv(PATH)
df_raw["Date"] = pd.to_datetime(df_raw["Date"])
df = (df_raw.rename(columns={"Local Site Name":"unique_id","Date":"ds"})
            .sort_values(["unique_id","ds"])
            .reset_index(drop=True))

# Spatial in km
lat = df["Site Latitude"].astype(float)
lon = df["Site Longitude"].astype(float)
df["y_km"] = lat * 111.32
df["x_km"] = lon * 111.32 * np.cos(np.deg2rad(lat))

# Weekly time index
t0 = df["ds"].min()
df["t_week"] = (df["ds"] - t0).dt.days / 7.0
df["t_idx"]  = (df["t_week"] - df["t_week"].min()).round().astype(int)

# Same-pollutant lags 1..3
for p in POLLUTANTS:
    for lag in [1,2,3]:
        df[f"{p}_lag_{lag}"] = df.groupby("unique_id")[p].shift(lag)

print("Non-NA counts (raw):")
for p in POLLUTANTS:
    print(f"  {p}: {int(df[p].notna().sum())}")

# ============================================================
# Variogram → bounds (fast hints)
# ============================================================
def make_residuals(df, pollutant, demean_by_site=True):
    dfx = df[["unique_id","ds","t_idx","x_km","y_km",pollutant]].dropna(subset=[pollutant]).copy()
    y = dfx[pollutant].values.astype(float)
    if demean_by_site:
        site_mean = dfx.groupby("unique_id")[pollutant].transform("mean").values
        y = y - site_mean
    dfx["resid"] = y
    return dfx

def temporal_variogram(dfr, max_lag_weeks=52, min_pairs=MIN_PAIRS_PER_BIN):
    outs = []
    for tau in range(1, max_lag_weeks+1):
        sq_diffs = []
        for _, g in dfr.groupby("unique_id", sort=False):
            gi = g.set_index("t_idx")["resid"]
            g2 = gi.reindex(gi.index + tau).dropna()
            gi2= gi.reindex(g2.index - tau).dropna()
            if len(gi2) and len(g2):
                sq_diffs.append((gi2.values - g2.values)**2)
        if sq_diffs:
            sd = np.concatenate(sq_diffs)
            if len(sd) >= min_pairs:
                outs.append((tau, 0.5 * float(np.mean(sd)), len(sd)))
    return pd.DataFrame(outs, columns=["lag_weeks","semivar","n_pairs"])

def spatial_variogram(dfr, bin_w=10.0, max_dist=500.0,
                      max_weeks=30, max_pairs_per_week=3000, min_pairs=MIN_PAIRS_PER_BIN):
    uniq_weeks = dfr["ds"].drop_duplicates().sort_values().values
    if len(uniq_weeks) > max_weeks:
        uniq_weeks = rng.choice(uniq_weeks, size=max_weeks, replace=False)
    edges = np.arange(0.0, max_dist + bin_w, bin_w)
    nb = len(edges) - 1
    sum_sq = np.zeros(nb); cnt = np.zeros(nb, int)
    for w in uniq_weeks:
        gw = dfr[dfr["ds"]==w]
        if len(gw) < 2: continue
        X = gw[["x_km","y_km"]].values
        Y = gw["resid"].values
        n = len(gw); m = n*(n-1)//2
        if m <= max_pairs_per_week:
            pairs = list(combinations(range(n), 2))
        else:
            pairs = set()
            while len(pairs) < max_pairs_per_week:
                i = rng.randint(0, n-1); j = rng.randint(i+1, n)
                pairs.add((i,j))
            pairs = list(pairs)
        Xi = X[[i for i,j in pairs]]; Xj = X[[j for i,j in pairs]]
        d  = np.sqrt(np.sum((Xi - Xj)**2, axis=1))
        dy = Y[[i for i,j in pairs]] - Y[[j for i,j in pairs]]
        sd = 0.5*(dy*dy)
        b  = np.floor(d/bin_w).astype(int)
        mask=(b>=0)&(b<nb)
        for k,val in zip(b[mask], sd[mask]): sum_sq[k]+=val; cnt[k]+=1

    centers = 0.5*(edges[:-1]+edges[1:])
    semivar = np.divide(sum_sq, cnt, out=np.full_like(sum_sq, np.nan, dtype=float), where=cnt>0)
    sv = pd.DataFrame({"dist_km":centers,"semivar":semivar,"n_pairs":cnt})
    sv = sv[(sv["n_pairs"]>=min_pairs) & np.isfinite(sv["semivar"])].reset_index(drop=True)
    return sv

def practical_range(df_v, sill, frac=0.95):
    if df_v is None or df_v.empty: return np.nan
    x = df_v.iloc[:,0].values.astype(float)
    y = np.maximum.accumulate(df_v["semivar"].values.astype(float))
    target = frac * sill
    if np.all(y < target): return float(x[-1])
    i = np.searchsorted(y, target)
    if i == 0: return float(x[0])
    x0,y0 = x[i-1], y[i-1]; x1,y1 = x[i], y[i]
    if y1 == y0: return float(x1)
    return float(x0 + (x1-x0)*(target-y0)/(y1-y0))

def rbf_l_from_r95(r95):
    return r95 / sqrt(2.0*log(20.0)) if (np.isfinite(r95) and r95>0) else np.nan

def matern32_l_from_r95(r95):
    if not (np.isfinite(r95) and r95>0): return np.nan
    f = lambda u: (1.0+u)*np.exp(-u) - 0.05
    try: u95 = brentq(f, 1e-6, 20.0)
    except Exception: u95 = 3.0
    return r95 / u95

def avg_site_acf_at_lag(dfr, lag=52):
    vals = []
    for _, g in dfr.groupby("unique_id", sort=False):
        x = g.sort_values("t_idx")["resid"].values
        if len(x) > lag+5:
            xc = x - x.mean()
            num = np.dot(xc[:-lag], xc[lag:])
            den = np.dot(xc, xc)
            if den>0: vals.append(num/den)
    return float(np.mean(vals)) if vals else np.nan

def bounds_from_est(l_est, factor=2.0, lo_floor=1e-2, hi_cap=1e3):
    if np.isfinite(l_est) and l_est>0:
        return (max(lo_floor, l_est/factor), min(hi_cap, l_est*factor))
    return (lo_floor, hi_cap)

def variogram_kernel_suggestions(df, pollutants=POLLUTANTS):
    suggestions = {}
    rows = []
    for p in pollutants:
        dfr = make_residuals(df, p, demean_by_site=DEMEAN_BY_SITE)
        if dfr.empty: 
            continue
        sill = float(np.var(dfr["resid"].values, ddof=1))
        tv = temporal_variogram(dfr, max_lag_weeks=MAX_TEMP_LAG_WEEKS, min_pairs=MIN_PAIRS_PER_BIN)
        sv = spatial_variogram(dfr, bin_w=SPATIAL_BIN_WIDTH_KM, max_dist=MAX_DISTANCE_KM,
                               max_weeks=MAX_WEEKS_FOR_SPATIAL, max_pairs_per_week=MAX_PAIRS_PER_WEEK,
                               min_pairs=MIN_PAIRS_PER_BIN)
        r95_t = practical_range(tv[["lag_weeks","semivar"]], sill) if not tv.empty else np.nan
        r95_s = practical_range(sv[["dist_km","semivar"]],   sill) if not sv.empty else np.nan

        l_rbf_t, l_m32_t = rbf_l_from_r95(r95_t), matern32_l_from_r95(r95_t)
        l_rbf_s, l_m32_s = rbf_l_from_r95(r95_s), matern32_l_from_r95(r95_s)

        b_rbf_t = bounds_from_est(l_rbf_t, factor=2.0, lo_floor=0.05, hi_cap=200.0)
        b_m32_t = bounds_from_est(l_m32_t, factor=2.0, lo_floor=0.05, hi_cap=200.0)
        b_rbf_s = bounds_from_est(l_rbf_s, factor=2.0, lo_floor=5.0,  hi_cap=500.0)
        b_m32_s = bounds_from_est(l_m32_s, factor=2.0, lo_floor=5.0,  hi_cap=500.0)

        acf52 = avg_site_acf_at_lag(dfr, lag=PERIOD_WEEKS)
        suggestions[p] = dict(
            sill=sill, temporal_variogram=tv, spatial_variogram=sv,
            r95_time_weeks=r95_t, r95_space_km=r95_s,
            l_rbf_time=l_rbf_t, l_m32_time=l_m32_t,
            l_rbf_space=l_rbf_s, l_m32_space=l_m32_s,
            bounds_rbf_time=b_rbf_t, bounds_m32_time=b_m32_t,
            bounds_rbf_space=b_rbf_s, bounds_m32_space=b_m32_s,
            acf52=acf52, suggest_periodic=(acf52>=ACF52_THRESHOLD if np.isfinite(acf52) else False)
        )
        rows.append([p, sill, r95_t, r95_s, l_rbf_t, l_m32_t, l_rbf_s, l_m32_s,
                     b_rbf_t, b_m32_t, b_rbf_s, b_m32_s, acf52, suggestions[p]["suggest_periodic"]])
    table = pd.DataFrame(rows, columns=[
        "Pollutant","Sill","r95_time_weeks","r95_space_km",
        "l_rbf_time","l_m32_time","l_rbf_space","l_m32_space",
        "bounds_rbf_time","bounds_m32_time","bounds_rbf_space","bounds_m32_space",
        "ACF_lag52","Suggest_Periodic"
    ])
    return table, suggestions

print("\nComputing variogram-based kernel hints…")
var_table, sugg = variogram_kernel_suggestions(df, pollutants=POLLUTANTS)
print("\n=== Variogram-based suggestions ===")
print(var_table.to_string(index=False))

# ============================================================
# Variogram-guided kernel builder (per pollutant)
# ============================================================
def _mid(lo, hi): return (lo+hi)/2.0

def build_kernel_variogram_guided(p, idx_space, idx_time, idx_lags, sugg):
    s = sugg[p]
    # --- time ---
    lo_t, hi_t = s["bounds_m32_time"]
    if p in DELTA_TARGET:
        lo_t = min(lo_t, 0.2)
    K_time_short = SliceKernel(C(0.7,(1e-3,1e3)) * Matern(length_scale=_mid(lo_t,hi_t), nu=0.5,
                                                          length_scale_bounds=(lo_t,hi_t)), idx_time)
    # broader long-term drift
    K_time_long  = SliceKernel(C(0.5,(1e-3,1e3)) * Matern(length_scale=12.0, nu=1.5,
                                                          length_scale_bounds=(4.0, 104.0)), idx_time)
    # --- lags (ARD) ---
    K_lags = SliceKernel(C(1.2,(1e-3,1e3)) * RBF(length_scale=[1.0]*len(idx_lags),
                                                 length_scale_bounds=(0.05, 50.0)), idx_lags)
    # --- space (PM2.5 only by default) ---
    if p == "PM2.5":
        lo_s, hi_s = s["bounds_m32_space"]
        K_space = SliceKernel(C(0.8,(1e-3,1e3)) * Matern(length_scale=[_mid(lo_s,hi_s)]*2, nu=1.5,
                                                        length_scale_bounds=(lo_s, hi_s)), idx_space)
    else:
        K_space = None
    # --- periodic (optional) ---
    K_per = None
    if p == "PM2.5":
        K_per = SliceKernel(C(0.5,(1e-3,1e3)) * ExpSineSquared(length_scale=5.0, periodicity=52.0,
                                                               length_scale_bounds=(1.0, 20.0),
                                                               periodicity_bounds="fixed"), idx_time)
    # --- combine & noise ---
    k_noise = WhiteKernel(noise_level=0.05, noise_level_bounds=(1e-5, 1.0))
    if p == "PM2.5":
        k = (K_space * K_time_long) + K_time_short + K_lags + (K_per if K_per is not None else 0) + k_noise
    else:
        k = K_time_short + 0.3**2 * K_time_long + K_lags + k_noise
    return k

def make_gp(kernel, restarts):
    return Pipeline([
        ("scale", StandardScaler()),
        ("gpr", GaussianProcessRegressor(
            kernel=kernel,
            optimizer=lbfgs_capped,
            n_restarts_optimizer=restarts,
            normalize_y=True,
            copy_X_train=False,
            random_state=RANDOM_STATE
        ))
    ])

# ============================================================
# Validation (quick)
# ============================================================
results = []
kernels_fitted = {}
overall_t0 = time.perf_counter()

for p in POLLUTANTS:
    print(f"\n=== {p} ===")
    # features per pollutant
    if p in DELTA_TARGET:
        lag_cols = [f"{p}_lag_2"] if USE_LAG2_ONLY_FOR_DELTA else [f"{p}_lag_2", f"{p}_lag_3"]
        features = ["x_km","y_km","t_week"] + lag_cols
    else:
        lag_cols = [f"{p}_lag_1", f"{p}_lag_2", f"{p}_lag_3"]
        features = ["x_km","y_km","t_week"] + lag_cols

    use_cols = ["unique_id","ds", p] + features + [f"{p}_lag_1"]
    dfi = df[use_cols].dropna(subset=[p] + features)
    if len(dfi) < 30:
        print(f"[{p}] not enough rows; skipping.")
        continue

    train_df, test_df = temporal_holdout_per_site(dfi, test_frac=0.2)
    train_df_sub = thin_training(train_df, stride=STRIDE_PER[p], max_total=MAX_TRAIN_PER[p], recent_weeks=RECENT_WEEKS)

    X_train_full = train_df_sub[features].values
    y_train_raw  = train_df_sub[p].values
    X_test_full  = test_df[features].values
    y_test_raw   = test_df[p].values

    if DEMEAN_BY_SITE:
        site_mean_train = train_df.groupby("unique_id")[p].mean()
        global_mean_train = float(site_mean_train.mean())
        y_train_res = y_train_raw - train_df_sub["unique_id"].map(site_mean_train).fillna(global_mean_train).values
        y_test_res  = y_test_raw  - test_df["unique_id"].map(site_mean_train).fillna(global_mean_train).values
    else:
        y_train_res, y_test_res = y_train_raw.copy(), y_test_raw.copy()
        site_mean_train, global_mean_train = None, 0.0

    if p in DELTA_TARGET:
        l1_tr = train_df_sub[f"{p}_lag_1"].values
        l1_te = test_df[f"{p}_lag_1"].values
        keep_tr = ~np.isnan(l1_tr); keep_te = ~np.isnan(l1_te)
        X_train = X_train_full[keep_tr]; y_train_res = y_train_res[keep_tr]; l1_tr = l1_tr[keep_tr]
        X_test  = X_test_full [keep_te]; y_test_res  = y_test_res [keep_te]; y_test_raw_sub = y_test_raw[keep_te]; l1_te = l1_te[keep_te]
        y_train_tgt = y_train_res - l1_tr; y_test_tgt = y_test_res - l1_te
    else:
        X_train, X_test = X_train_full, X_test_full
        y_train_tgt, y_test_tgt = y_train_res, y_test_res
        y_test_raw_sub = y_test_raw

    tf = Y_TRANSFORM.get(p, 'none')
    if tf == 'yeojohnson':
        pt = PowerTransformer(method='yeo-johnson', standardize=False)
        y_train_t = pt.fit_transform(y_train_tgt.reshape(-1,1)).ravel()
        y_test_t  = pt.transform(y_test_tgt.reshape(-1,1)).ravel()
        inv = lambda z: pt.inverse_transform(np.asarray(z).reshape(-1,1)).ravel()
    else:
        y_train_t = y_train_tgt.copy()
        y_test_t  = y_test_tgt.copy()
        inv = lambda z: np.asarray(z)

    idx_space, idx_time = (0,1), (2,)
    idx_lags = list(range(3, 3+len(lag_cols)))
    kernel = build_kernel_variogram_guided(p, idx_space, idx_time, idx_lags, sugg)
    gp = make_gp(kernel, restarts=RESTARTS_PER[p])

    t0 = time.perf_counter(); gp.fit(X_train, y_train_t); fit_secs = time.perf_counter() - t0
    mu_t, std_t = gp.predict(X_test, return_std=True)
    nlpd = float(np.mean(nlpds(y_test_t, mu_t, std_t)))

    mu_residual_space = inv(mu_t)
    if p in DELTA_TARGET: mu_residual_space = mu_residual_space + l1_te
    if DEMEAN_BY_SITE and site_mean_train is not None:
        mu = mu_residual_space + test_df.loc[test_df.index[keep_te] if p in DELTA_TARGET else test_df.index, "unique_id"]\
                                   .map(site_mean_train).fillna(global_mean_train).values
    else:
        mu = mu_residual_space

    rmse = float(np.sqrt(mean_squared_error(y_test_raw_sub, mu)))
    mae  = float(mean_absolute_error(y_test_raw_sub, mu))
    if p in DELTA_TARGET:
        y_pred_base = np.asarray(l1_te, dtype=float).ravel()
    else:
        y_pred_base = np.asarray(test_df[f"{p}_lag_1"].values, dtype=float).ravel()
    y_true_base = np.asarray(y_test_raw_sub, dtype=float).ravel()
    n = min(len(y_true_base), len(y_pred_base))
    mask_base = np.isfinite(y_true_base[:n]) & np.isfinite(y_pred_base[:n])
    if mask_base.sum() == 0:
        rmse_persist = np.nan; skill = np.nan
    else:
        rmse_persist = float(np.sqrt(mean_squared_error(y_true_base[:n][mask_base], y_pred_base[:n][mask_base])))
        skill = 1 - rmse / rmse_persist if rmse_persist > 0 else np.nan
    print(f"[{p}] fit={fit_secs:.1f}s | RMSE={rmse:.4f} | MAE={mae:.4f} | skill_vs_persist={skill:.3f} | NLPD={nlpd:.3f}")

    kernels_fitted[p] = gp.named_steps["gpr"].kernel_
    results.append([p, len(X_train), len(X_test), rmse, mae, rmse_persist, skill, nlpd])

print("\nValidation metrics:")
print(pd.DataFrame(results, columns=["Pollutant","n_train","n_test","RMSE","MAE","RMSE_persist","Skill","NLPD"]).to_string(index=False))

# ============================================================
# PM2.5 — Full fit on observed + site-level imputation with variance
# ============================================================
POLLUTANT = "PM2.5"
features_pm = ["x_km","y_km","t_week", f"{POLLUTANT}_lag_1", f"{POLLUTANT}_lag_2", f"{POLLUTANT}_lag_3"]
all_cols = ["unique_id","ds","County","Site Longitude","Site Latitude", POLLUTANT] + features_pm
dfa = df[all_cols].copy()

obs_mask  = dfa[POLLUTANT].notna()
feat_mask = dfa[features_pm].notna().all(axis=1)
train_all = dfa[obs_mask & feat_mask].copy()
missing   = dfa[~obs_mask & feat_mask].copy()

# De-mean by site (residual target)
site_mean_all = train_all.groupby("unique_id")[POLLUTANT].mean()
global_mean_all = float(site_mean_all.mean())

X_train_all = train_all[features_pm].values
y_train_all = train_all[POLLUTANT].values
y_train_all_res = y_train_all - train_all["unique_id"].map(site_mean_all).fillna(global_mean_all).values

# Transform target
if PM_TF == "yeojohnson":
    pt_all = PowerTransformer(method="yeo-johnson", standardize=False)
    y_train_all_t = pt_all.fit_transform(y_train_all_res.reshape(-1,1)).ravel()
    inv_all = lambda z: pt_all.inverse_transform(np.asarray(z).reshape(-1,1)).ravel()
else:
    y_train_all_t = y_train_all_res.copy()
    inv_all = lambda z: np.asarray(z)

# Kernel for PM2.5 (reuse the variogram-guided builder)
idx_space, idx_time, idx_lags = (0,1), (2,), (3,4,5)
kernel_pm = build_kernel_variogram_guided(POLLUTANT, idx_space, idx_time, idx_lags, sugg)
gp_all = make_gp(kernel_pm, restarts=RESTARTS_PER[POLLUTANT])

# Thin training for speed if desired
train_all_sub = thin_training(train_all, stride=STRIDE_ALL, max_total=MAX_TRAIN_ALL, recent_weeks=RECENT_WEEKS_ALL)
X_train_all_sub = train_all_sub[features_pm].values
y_train_all_sub = (train_all_sub[POLLUTANT].values
                   - train_all_sub["unique_id"].map(site_mean_all).fillna(global_mean_all).values)
if PM_TF == "yeojohnson":
    y_train_all_sub_t = pt_all.transform(y_train_all_sub.reshape(-1,1)).ravel()
else:
    y_train_all_sub_t = y_train_all_sub.copy()

t0 = time.perf_counter()
gp_all.fit(X_train_all_sub, y_train_all_sub_t)
print(f"\n[PM2.5] Full fit done in {time.perf_counter()-t0:.1f}s")
print("Fitted kernel:\n", gp_all.named_steps["gpr"].kernel_)

# Site-level predictions (missing rows) with variance
if len(missing):
    X_miss = missing[features_pm].values
    mu_t, std_t = gp_all.predict(X_miss, return_std=True)  # in transformed residual space
    # variance in transformed space
    var_t = std_t**2
    # back-transform mean & variance via delta method
    if PM_TF == "yeojohnson":
        eps = 1e-4
        mu_res = inv_all(mu_t)
        up = inv_all(mu_t + eps); dn = inv_all(mu_t - eps)
        gprime = (up - dn) / (2*eps)
        var_res = (gprime**2) * var_t
    else:
        mu_res  = mu_t.copy()
        var_res = var_t.copy()
    site_shift = missing["unique_id"].map(site_mean_all).fillna(global_mean_all).values
    mu_all = mu_res + site_shift
    out_missing = missing.copy()
    out_missing[f"{POLLUTANT}_pred"] = mu_all
    out_missing[f"{POLLUTANT}_var"]  = var_res
    out_missing["is_imputed"] = True
else:
    out_missing = dfa.iloc[0:0].copy()

# Observed rows (keep their value; variance = 0 if you want them as known)
out_observed = dfa[obs_mask].copy()
out_observed[f"{POLLUTANT}_pred"] = out_observed[POLLUTANT].values
out_observed[f"{POLLUTANT}_var"]  = 0.0
out_observed["is_imputed"] = False

out_sites = pd.concat([out_observed, out_missing], ignore_index=True, sort=False)
out_sites = out_sites.sort_values(["unique_id","ds"]).reset_index(drop=True)

# Save site-level output
site_keep = ["ds","unique_id","County","Site Longitude","Site Latitude",
             POLLUTANT, f"{POLLUTANT}_pred", f"{POLLUTANT}_var","is_imputed"] + features_pm
out_sites = out_sites[site_keep]
out_sites.to_csv("pm25_2023_imputed_with_var.csv", index=False)
print(f"Wrote site-level file with variance: pm25_2023_imputed_with_var.csv")
print(f"Imputed rows: {int(out_sites['is_imputed'].sum())}")

# ============================================================
# County-week aggregation using GP joint covariance (MVE/GLS)
# ============================================================
from sklearn.gaussian_process.kernels import WhiteKernel

def _white_variance_from_kernel(k):
    """Sum WhiteKernel noise variances present in the fitted kernel."""
    if isinstance(k, WhiteKernel):
        return float(k.noise_level)
    total = 0.0
    for child in ("k1","k2"):
        if hasattr(k, child):
            total += _white_variance_from_kernel(getattr(k, child))
    return total

def county_week_from_gp(gp_pipeline, dfa, ds_week, county_name,
                        features, site_mean_all, global_mean_all,
                        transform="yeojohnson", inv_all=None,
                        include_obs_noise=True, ridge=1e-8):
    """Return MVE (GLS) county mean/var using joint predictive covariance from the GP."""
    sub = dfa[(dfa["ds"] == ds_week) & (dfa["County"] == county_name)].copy()
    sub = sub[sub[features].notna().all(axis=1)]
    n = len(sub)
    if n == 0:
        return None

    X = sub[features].values

    # Joint posterior for the pipeline (handles scaling internally)
    mu_t, Sigma_t = gp_pipeline.predict(X, return_cov=True)

    # Back-transform to residual space
    if transform == "yeojohnson":
        if inv_all is None:
            raise ValueError("inv_all required when transform='yeojohnson'.")
        eps = 1e-4
        mu_res = inv_all(mu_t)
        up = inv_all(mu_t + eps); dn = inv_all(mu_t - eps)
        gprime = (up - dn) / (2*eps)
        J = gprime[:, None]
        Sigma_res = (J * Sigma_t) * gprime[None, :]
    else:
        mu_res = mu_t.copy()
        Sigma_res = Sigma_t.copy()

    # Optionally remove observation noise from diagonal (latent field)
    if not include_obs_noise:
        sigma2_noise = _white_variance_from_kernel(gp_pipeline.named_steps["gpr"].kernel_)
        Sigma_res = Sigma_res.copy()
        np.fill_diagonal(Sigma_res, np.clip(np.diag(Sigma_res) - sigma2_noise, 0.0, np.inf))

    # Add back site means
    mu_sites = mu_res + sub["unique_id"].map(site_mean_all).fillna(global_mean_all).values

    # GLS / MVE weights: w ∝ Σ^{-1} 1, unbiased => sum(w)=1
    ones = np.ones(n)
    # ridge for stability
    Sigma_r = Sigma_res + ridge*np.eye(n)
    x = np.linalg.solve(Sigma_r, ones)
    denom = float(ones @ x)
    w = x / denom
    county_mean = float(w @ mu_sites)
    county_var  = float(1.0 / denom)

    return {
        "County": county_name,
        "ds": ds_week,
        "n_sites": n,
        "PM2.5_county_mean": county_mean,
        "PM2.5_county_var": max(county_var, 0.0)
    }

# Aggregate all county-weeks present in data (with features available)
groups = (dfa[features_pm + ["County","ds"]]
          .dropna(subset=features_pm)
          .groupby(["County","ds"]).size().reset_index()[["County","ds"]])

rows = []
for _, row in groups.iterrows():
    res = county_week_from_gp(
        gp_pipeline=gp_all,
        dfa=dfa,
        ds_week=row["ds"],
        county_name=row["County"],
        features=features_pm,
        site_mean_all=site_mean_all,
        global_mean_all=global_mean_all,
        transform=PM_TF,
        inv_all=inv_all,
        include_obs_noise=INCLUDE_OBS_NOISE,
        ridge=RIDGE
    )
    if res is not None:
        rows.append(res)

county_df = pd.DataFrame(rows).sort_values(["County","ds"]).reset_index(drop=True)
county_df.to_csv("pm25_2023_by_county_week_cov.csv", index=False)
print(f"Wrote county-week file (MVE with covariance): pm25_2023_by_county_week_cov.csv")
print(county_df.head(10))


Non-NA counts (raw):
  PM2.5: 8167
  NO2: 4936
  CO: 2481

Computing variogram-based kernel hints…

=== Variogram-based suggestions ===
Pollutant      Sill  r95_time_weeks  r95_space_km  l_rbf_time  l_m32_time  l_rbf_space  l_m32_space                          bounds_rbf_time                           bounds_m32_time                        bounds_rbf_space                        bounds_m32_space  ACF_lag52  Suggest_Periodic
    PM2.5 38.996110        4.469509     59.163565    1.825969    0.942166    24.170622    12.471597 (0.9129843687570954, 3.6519374750283817) (0.47108314042755806, 1.8843325617102322)  (12.08531123618204, 48.34124494472816) (6.235798295140136, 24.943193180560545)        NaN             False
      NO2 41.588073       10.813931    495.000000    4.417912    2.279562   202.226796   104.345307  (2.2089562080409113, 8.835824832163645)   (1.1397807708006016, 4.559123083202406) (101.11339820677465, 404.4535928270986) (52.17265354870779, 208.69061419483117)        NaN       

# 2024

In [3]:

# ============================================
# Spatio-temporal GP (validation) + PM2.5 full fit/imputation — 2024
# + County-week aggregation using joint covariance (MVE/GLS)
# ============================================
import time, warnings
import numpy as np
import pandas as pd
from itertools import combinations
from math import sqrt, log
from scipy.optimize import fmin_l_bfgs_b, brentq

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    Kernel, RBF, Matern, ExpSineSquared, WhiteKernel, ConstantKernel as C
)
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------
# Quiet noisy warnings
# -----------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------
# Global Config (2024)
# -----------------------------
PATH = "full_2024_weekly_averages.csv"   # <<---- 2024 file
RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)

POLLUTANTS = ["PM2.5", "NO2", "CO"]
DEMEAN_BY_SITE = True
Y_TRANSFORM = {'PM2.5': 'yeojohnson', 'NO2': 'yeojohnson', 'CO': 'none'}

# Validation
MAX_TRAIN_PER = {'PM2.5': 1000, 'NO2': 1000, 'CO': 1000}
STRIDE_PER    = {'PM2.5': 1,    'NO2': 1,    'CO': 1}
RECENT_WEEKS  = 52
MAXITER_LBFGS = 20
RESTARTS_PER  = {'PM2.5': 1, 'NO2': 1, 'CO': 1}
DELTA_TARGET = {'NO2', 'CO'}
USE_LAG2_ONLY_FOR_DELTA = True

# Periodicity probe (used for bounds only)
PERIOD_WEEKS   = 52
ACF52_THRESHOLD = 0.2

# Variogram settings
MIN_PAIRS_PER_BIN    = 100
MAX_TEMP_LAG_WEEKS   = 52
SPATIAL_BIN_WIDTH_KM = 10.0
MAX_DISTANCE_KM      = 500.0
MAX_WEEKS_FOR_SPATIAL= 30
MAX_PAIRS_PER_WEEK   = 3000

# Full PM2.5 fit/impute
PM_TF = "yeojohnson"     # target transform for PM2.5 ('yeojohnson' or 'none')
MAX_TRAIN_ALL = 3000
RECENT_WEEKS_ALL = 52
STRIDE_ALL = 1

# County-week covariance aggregation
INCLUDE_OBS_NOISE = True   # True => keep obs noise in Σ; False => subtract WhiteKernel variance
RIDGE = 1e-8               # diag ridge for GLS/MVE stability

# -----------------------------
# Optimizer wrapper (capped L-BFGS-B)
# -----------------------------
def lbfgs_capped(obj_func, initial_theta, bounds):
    theta_opt, fmin, _ = fmin_l_bfgs_b(
        obj_func, initial_theta, bounds=bounds, maxiter=MAXITER_LBFGS
    )
    return theta_opt, fmin

# -----------------------------
# SliceKernel (feature selector)
# -----------------------------
class SliceKernel(Kernel):
    def __init__(self, base_kernel, active_dims):
        self.base_kernel = base_kernel
        self.active_dims = tuple(active_dims)
    def __call__(self, X, Y=None, eval_gradient=False):
        idx = np.asarray(self.active_dims)
        Xs = X[:, idx]
        Ys = None if Y is None else Y[:, idx]
        return self.base_kernel(Xs, Ys, eval_gradient=eval_gradient)
    def diag(self, X):
        idx = np.asarray(self.active_dims)
        return self.base_kernel.diag(X[:, idx])
    def is_stationary(self):
        return getattr(self.base_kernel, "is_stationary", lambda: False)()
    def get_params(self, deep=True):
        params = {"base_kernel": self.base_kernel, "active_dims": self.active_dims}
        if deep and hasattr(self.base_kernel, "get_params"):
            for k, v in self.base_kernel.get_params(deep=True).items():
                params[f"base_kernel__{k}"] = v
        return params
    def set_params(self, **params):
        base_params = {k.split("__",1)[1]: v for k,v in params.items() if k.startswith("base_kernel__")}
        if base_params and hasattr(self.base_kernel, "set_params"):
            self.base_kernel.set_params(**base_params)
        if "base_kernel" in params: self.base_kernel = params["base_kernel"]
        if "active_dims" in params: self.active_dims = tuple(params["active_dims"])
        return self
    @property
    def hyperparameters(self): return self.base_kernel.hyperparameters
    @property
    def theta(self): return self.base_kernel.theta
    @theta.setter
    def theta(self, t): self.base_kernel.theta = t
    @property
    def bounds(self): return self.base_kernel.bounds
    def clone_with_theta(self, theta): return SliceKernel(self.base_kernel.clone_with_theta(theta), self.active_dims)
    def __repr__(self): return f"SliceKernel(base_kernel={self.base_kernel!r}, active_dims={self.active_dims!r})"

# -----------------------------
# Utilities
# -----------------------------
def temporal_holdout_per_site(dfi, group_col='unique_id', date_col='ds', test_frac=0.2):
    dfi = dfi.sort_values([group_col, date_col]).reset_index(drop=True)
    is_test = np.zeros(len(dfi), dtype=bool)
    for _, idx in dfi.groupby(group_col).indices.items():
        k = max(1, int(np.ceil(test_frac * len(idx))))
        is_test[idx[-k:]] = True
    return dfi[~is_test].copy(), dfi[is_test].copy()

def thin_training(train_df, stride, max_total, recent_weeks, random_state=RANDOM_STATE,
                  min_per_site=5, frac_recent=0.7):
    rng = np.random.RandomState(random_state)
    if (max_total is None) and (stride == 1):
        return train_df
    parts_recent, parts_older = [], []
    for uid, g in train_df.groupby('unique_id', sort=False):
        g = g.sort_values('t_week', kind='mergesort')
        cutoff = g['t_week'].max() - recent_weeks
        gr = g[g['t_week'] >= cutoff]
        go = g[g['t_week'] <  cutoff]
        if stride > 1:
            if len(gr) > 0: gr = gr.iloc[::stride]
            if len(go) > 0: go = go.iloc[::stride]
        parts_recent.append(gr); parts_older.append(go)
    recent = pd.concat(parts_recent, ignore_index=False) if parts_recent else train_df.iloc[0:0]
    older  = pd.concat(parts_older,  ignore_index=False) if parts_older  else train_df.iloc[0:0]
    if max_total is None:
        return pd.concat([recent, older], ignore_index=True)
    site_counts = train_df['unique_id'].value_counts()
    quotas = (site_counts / site_counts.sum() * max_total).round().astype(int).clip(lower=min_per_site)
    chosen = []
    for uid, q in quotas.items():
        gr = recent[recent['unique_id'] == uid]; go = older[older['unique_id'] == uid]
        q_recent = int(np.ceil(q * frac_recent)); q_older  = q - q_recent
        pick_r = gr if len(gr) <= q_recent else gr.sample(q_recent, random_state=rng)
        pick_o = go if len(go) <= q_older  else go.sample(q_older,  random_state=rng)
        chosen.append(pd.concat([pick_r, pick_o], ignore_index=False))
    sub = pd.concat(chosen, ignore_index=True)
    if len(sub) > max_total: sub = sub.sample(max_total, random_state=rng)
    return sub

def nlpds(y_t_true, mu_t, sigma_t):
    sigma2 = np.maximum(sigma_t**2, 1e-12)
    return 0.5*np.log(2*np.pi*sigma2) + 0.5*((y_t_true - mu_t)**2)/sigma2

def find_slice_by_dims(k, dims):
    if isinstance(k, SliceKernel) and tuple(k.active_dims) == tuple(dims):
        return k
    for child in ("k1","k2"):
        if hasattr(k, child):
            ret = find_slice_by_dims(getattr(k, child), dims)
            if ret is not None: return ret
    return None

# -----------------------------
# Load data & features (2024 file)
# -----------------------------
df_raw = pd.read_csv(PATH)
df_raw["Date"] = pd.to_datetime(df_raw["Date"])
df = (df_raw.rename(columns={"Local Site Name":"unique_id","Date":"ds"})
            .sort_values(["unique_id","ds"])
            .reset_index(drop=True))

# Spatial in km
lat = df["Site Latitude"].astype(float)
lon = df["Site Longitude"].astype(float)
df["y_km"] = lat * 111.32
df["x_km"] = lon * 111.32 * np.cos(np.deg2rad(lat))

# Weekly time index
t0 = df["ds"].min()
df["t_week"] = (df["ds"] - t0).dt.days / 7.0
df["t_idx"]  = (df["t_week"] - df["t_week"].min()).round().astype(int)

# Same-pollutant lags 1..3
for p in POLLUTANTS:
    for lag in [1,2,3]:
        df[f"{p}_lag_{lag}"] = df.groupby("unique_id")[p].shift(lag)

print("Non-NA counts (raw):")
for p in POLLUTANTS:
    print(f"  {p}: {int(df[p].notna().sum())}")

# ============================================================
# Variogram → bounds (fast hints)
# ============================================================
def make_residuals(df, pollutant, demean_by_site=True):
    dfx = df[["unique_id","ds","t_idx","x_km","y_km",pollutant]].dropna(subset=[pollutant]).copy()
    y = dfx[pollutant].values.astype(float)
    if demean_by_site:
        site_mean = dfx.groupby("unique_id")[pollutant].transform("mean").values
        y = y - site_mean
    dfx["resid"] = y
    return dfx

def temporal_variogram(dfr, max_lag_weeks=52, min_pairs=MIN_PAIRS_PER_BIN):
    outs = []
    for tau in range(1, max_lag_weeks+1):
        sq_diffs = []
        for _, g in dfr.groupby("unique_id", sort=False):
            gi = g.set_index("t_idx")["resid"]
            g2 = gi.reindex(gi.index + tau).dropna()
            gi2= gi.reindex(g2.index - tau).dropna()
            if len(gi2) and len(g2):
                sq_diffs.append((gi2.values - g2.values)**2)
        if sq_diffs:
            sd = np.concatenate(sq_diffs)
            if len(sd) >= min_pairs:
                outs.append((tau, 0.5 * float(np.mean(sd)), len(sd)))
    return pd.DataFrame(outs, columns=["lag_weeks","semivar","n_pairs"])

def spatial_variogram(dfr, bin_w=10.0, max_dist=500.0,
                      max_weeks=30, max_pairs_per_week=3000, min_pairs=MIN_PAIRS_PER_BIN):
    uniq_weeks = dfr["ds"].drop_duplicates().sort_values().values
    if len(uniq_weeks) > max_weeks:
        uniq_weeks = rng.choice(uniq_weeks, size=max_weeks, replace=False)
    edges = np.arange(0.0, max_dist + bin_w, bin_w)
    nb = len(edges) - 1
    sum_sq = np.zeros(nb); cnt = np.zeros(nb, int)
    for w in uniq_weeks:
        gw = dfr[dfr["ds"]==w]
        if len(gw) < 2: continue
        X = gw[["x_km","y_km"]].values
        Y = gw["resid"].values
        n = len(gw); m = n*(n-1)//2
        if m <= max_pairs_per_week:
            pairs = list(combinations(range(n), 2))
        else:
            pairs = set()
            while len(pairs) < max_pairs_per_week:
                i = rng.randint(0, n-1); j = rng.randint(i+1, n)
                pairs.add((i,j))
            pairs = list(pairs)
        Xi = X[[i for i,j in pairs]]; Xj = X[[j for i,j in pairs]]
        d  = np.sqrt(np.sum((Xi - Xj)**2, axis=1))
        dy = Y[[i for i,j in pairs]] - Y[[j for i,j in pairs]]
        sd = 0.5*(dy*dy)
        b  = np.floor(d/bin_w).astype(int)
        mask=(b>=0)&(b<nb)
        for k,val in zip(b[mask], sd[mask]): sum_sq[k]+=val; cnt[k]+=1
    centers = 0.5*(edges[:-1]+edges[1:])
    semivar = np.divide(sum_sq, cnt, out=np.full_like(sum_sq, np.nan, dtype=float), where=cnt>0)
    sv = pd.DataFrame({"dist_km":centers,"semivar":semivar,"n_pairs":cnt})
    sv = sv[(sv["n_pairs"]>=min_pairs) & np.isfinite(sv["semivar"])].reset_index(drop=True)
    return sv

def practical_range(df_v, sill, frac=0.95):
    if df_v is None or df_v.empty: return np.nan
    x = df_v.iloc[:,0].values.astype(float)
    y = np.maximum.accumulate(df_v["semivar"].values.astype(float))
    target = frac * sill
    if np.all(y < target): return float(x[-1])
    i = np.searchsorted(y, target)
    if i == 0: return float(x[0])
    x0,y0 = x[i-1], y[i-1]; x1,y1 = x[i], y[i]
    if y1 == y0: return float(x1)
    return float(x0 + (x1-x0)*(target-y0)/(y1-y0))

def rbf_l_from_r95(r95):
    return r95 / sqrt(2.0*log(20.0)) if (np.isfinite(r95) and r95>0) else np.nan

def matern32_l_from_r95(r95):
    if not (np.isfinite(r95) and r95>0): return np.nan
    f = lambda u: (1.0+u)*np.exp(-u) - 0.05
    try: u95 = brentq(f, 1e-6, 20.0)
    except Exception: u95 = 3.0
    return r95 / u95

def avg_site_acf_at_lag(dfr, lag=52):
    vals = []
    for _, g in dfr.groupby("unique_id", sort=False):
        x = g.sort_values("t_idx")["resid"].values
        if len(x) > lag+5:
            xc = x - x.mean()
            num = np.dot(xc[:-lag], xc[lag:])
            den = np.dot(xc, xc)
            if den>0: vals.append(num/den)
    return float(np.mean(vals)) if vals else np.nan

def bounds_from_est(l_est, factor=2.0, lo_floor=1e-2, hi_cap=1e3):
    if np.isfinite(l_est) and l_est>0:
        return (max(lo_floor, l_est/factor), min(hi_cap, l_est*factor))
    return (lo_floor, hi_cap)

def variogram_kernel_suggestions(df, pollutants=POLLUTANTS):
    suggestions = {}
    rows = []
    for p in pollutants:
        dfr = make_residuals(df, p, demean_by_site=DEMEAN_BY_SITE)
        if dfr.empty: 
            continue
        sill = float(np.var(dfr["resid"].values, ddof=1))
        tv = temporal_variogram(dfr, max_lag_weeks=MAX_TEMP_LAG_WEEKS, min_pairs=MIN_PAIRS_PER_BIN)
        sv = spatial_variogram(dfr, bin_w=SPATIAL_BIN_WIDTH_KM, max_dist=MAX_DISTANCE_KM,
                               max_weeks=MAX_WEEKS_FOR_SPATIAL, max_pairs_per_week=MAX_PAIRS_PER_WEEK,
                               min_pairs=MIN_PAIRS_PER_BIN)
        r95_t = practical_range(tv[["lag_weeks","semivar"]], sill) if not tv.empty else np.nan
        r95_s = practical_range(sv[["dist_km","semivar"]],   sill) if not sv.empty else np.nan
        l_rbf_t, l_m32_t = rbf_l_from_r95(r95_t), matern32_l_from_r95(r95_t)
        l_rbf_s, l_m32_s = rbf_l_from_r95(r95_s), matern32_l_from_r95(r95_s)
        b_rbf_t = bounds_from_est(l_rbf_t, factor=2.0, lo_floor=0.05, hi_cap=200.0)
        b_m32_t = bounds_from_est(l_m32_t, factor=2.0, lo_floor=0.05, hi_cap=200.0)
        b_rbf_s = bounds_from_est(l_rbf_s, factor=2.0, lo_floor=5.0,  hi_cap=500.0)
        b_m32_s = bounds_from_est(l_m32_s, factor=2.0, lo_floor=5.0,  hi_cap=500.0)
        acf52 = avg_site_acf_at_lag(dfr, lag=PERIOD_WEEKS)
        suggestions[p] = dict(
            sill=sill, temporal_variogram=tv, spatial_variogram=sv,
            r95_time_weeks=r95_t, r95_space_km=r95_s,
            l_rbf_time=l_rbf_t, l_m32_time=l_m32_t,
            l_rbf_space=l_rbf_s, l_m32_space=l_m32_s,
            bounds_rbf_time=b_rbf_t, bounds_m32_time=b_m32_t,
            bounds_rbf_space=b_rbf_s, bounds_m32_space=b_m32_s,
            acf52=acf52, suggest_periodic=(acf52>=ACF52_THRESHOLD if np.isfinite(acf52) else False)
        )
        rows.append([p, sill, r95_t, r95_s, l_rbf_t, l_m32_t, l_rbf_s, l_m32_s,
                     b_rbf_t, b_m32_t, b_rbf_s, b_m32_s, acf52, suggestions[p]["suggest_periodic"]])
    table = pd.DataFrame(rows, columns=[
        "Pollutant","Sill","r95_time_weeks","r95_space_km",
        "l_rbf_time","l_m32_time","l_rbf_space","l_m32_space",
        "bounds_rbf_time","bounds_m32_time","bounds_rbf_space","bounds_m32_space",
        "ACF_lag52","Suggest_Periodic"
    ])
    return table, suggestions

print("\nComputing variogram-based kernel hints…")
var_table, sugg = variogram_kernel_suggestions(df, pollutants=POLLUTANTS)
print("\n=== Variogram-based suggestions ===")
print(var_table.to_string(index=False))

# ============================================================
# Variogram-guided kernel builder (per pollutant)
# ============================================================
def _mid(lo, hi): return (lo+hi)/2.0

def build_kernel_variogram_guided(p, idx_space, idx_time, idx_lags, sugg):
    s = sugg[p]
    lo_t, hi_t = s["bounds_m32_time"]
    if p in DELTA_TARGET:
        lo_t = min(lo_t, 0.2)
    K_time_short = SliceKernel(C(0.7,(1e-3,1e3)) * Matern(length_scale=_mid(lo_t,hi_t), nu=0.5,
                                                          length_scale_bounds=(lo_t,hi_t)), idx_time)
    K_time_long  = SliceKernel(C(0.5,(1e-3,1e3)) * Matern(length_scale=12.0, nu=1.5,
                                                          length_scale_bounds=(4.0, 104.0)), idx_time)
    K_lags = SliceKernel(C(1.2,(1e-3,1e3)) * RBF(length_scale=[1.0]*len(idx_lags),
                                                 length_scale_bounds=(0.05, 50.0)), idx_lags)
    if p == "PM2.5":
        lo_s, hi_s = s["bounds_m32_space"]
        K_space = SliceKernel(C(0.8,(1e-3,1e3)) * Matern(length_scale=[_mid(lo_s,hi_s)]*2, nu=1.5,
                                                        length_scale_bounds=(lo_s, hi_s)), idx_space)
    else:
        K_space = None
    K_per = None
    if p == "PM2.5":
        K_per = SliceKernel(C(0.5,(1e-3,1e3)) * ExpSineSquared(length_scale=5.0, periodicity=52.0,
                                                               length_scale_bounds=(1.0, 20.0),
                                                               periodicity_bounds="fixed"), idx_time)
    k_noise = WhiteKernel(noise_level=0.05, noise_level_bounds=(1e-5, 1.0))
    if p == "PM2.5":
        k = (K_space * K_time_long) + K_time_short + K_lags + (K_per if K_per is not None else 0) + k_noise
    else:
        k = K_time_short + 0.3**2 * K_time_long + K_lags + k_noise
    return k

def make_gp(kernel, restarts):
    return Pipeline([
        ("scale", StandardScaler()),
        ("gpr", GaussianProcessRegressor(
            kernel=kernel,
            optimizer=lbfgs_capped,
            n_restarts_optimizer=restarts,
            normalize_y=True,
            copy_X_train=False,
            random_state=RANDOM_STATE
        ))
    ])

# ============================================================
# Validation (quick, on 2024 itself)
# ============================================================
results = []
kernels_fitted = {}
overall_t0 = time.perf_counter()

for p in POLLUTANTS:
    print(f"\n=== {p} ===")
    if p in DELTA_TARGET:
        lag_cols = [f"{p}_lag_2"] if USE_LAG2_ONLY_FOR_DELTA else [f"{p}_lag_2", f"{p}_lag_3"]
        features = ["x_km","y_km","t_week"] + lag_cols
    else:
        lag_cols = [f"{p}_lag_1", f"{p}_lag_2", f"{p}_lag_3"]
        features = ["x_km","y_km","t_week"] + lag_cols

    use_cols = ["unique_id","ds", p] + features + [f"{p}_lag_1"]
    dfi = df[use_cols].dropna(subset=[p] + features)
    if len(dfi) < 30:
        print(f"[{p}] not enough rows; skipping.")
        continue

    train_df, test_df = temporal_holdout_per_site(dfi, test_frac=0.2)
    train_df_sub = thin_training(train_df, stride=STRIDE_PER[p], max_total=MAX_TRAIN_PER[p], recent_weeks=RECENT_WEEKS)

    X_train_full = train_df_sub[features].values
    y_train_raw  = train_df_sub[p].values
    X_test_full  = test_df[features].values
    y_test_raw   = test_df[p].values

    if DEMEAN_BY_SITE:
        site_mean_train = train_df.groupby("unique_id")[p].mean()
        global_mean_train = float(site_mean_train.mean())
        y_train_res = y_train_raw - train_df_sub["unique_id"].map(site_mean_train).fillna(global_mean_train).values
        y_test_res  = y_test_raw  - test_df["unique_id"].map(site_mean_train).fillna(global_mean_train).values
    else:
        y_train_res, y_test_res = y_train_raw.copy(), y_test_raw.copy()
        site_mean_train, global_mean_train = None, 0.0

    if p in DELTA_TARGET:
        l1_tr = train_df_sub[f"{p}_lag_1"].values
        l1_te = test_df[f"{p}_lag_1"].values
        keep_tr = ~np.isnan(l1_tr); keep_te = ~np.isnan(l1_te)
        X_train = X_train_full[keep_tr]; y_train_res = y_train_res[keep_tr]; l1_tr = l1_tr[keep_tr]
        X_test  = X_test_full [keep_te]; y_test_res  = y_test_res [keep_te]; y_test_raw_sub = y_test_raw[keep_te]; l1_te = l1_te[keep_te]
        y_train_tgt = y_train_res - l1_tr; y_test_tgt = y_test_res - l1_te
    else:
        X_train, X_test = X_train_full, X_test_full
        y_train_tgt, y_test_tgt = y_train_res, y_test_res
        y_test_raw_sub = y_test_raw

    tf = Y_TRANSFORM.get(p, 'none')
    if tf == 'yeojohnson':
        pt = PowerTransformer(method='yeo-johnson', standardize=False)
        y_train_t = pt.fit_transform(y_train_tgt.reshape(-1,1)).ravel()
        y_test_t  = pt.transform(y_test_tgt.reshape(-1,1)).ravel()
        inv = lambda z: pt.inverse_transform(np.asarray(z).reshape(-1,1)).ravel()
    else:
        y_train_t = y_train_tgt.copy()
        y_test_t  = y_test_tgt.copy()
        inv = lambda z: np.asarray(z)

    idx_space, idx_time = (0,1), (2,)
    idx_lags = list(range(3, 3+len(lag_cols)))
    kernel = build_kernel_variogram_guided(p, idx_space, idx_time, idx_lags, sugg)
    gp = make_gp(kernel, restarts=RESTARTS_PER[p])

    t0 = time.perf_counter(); gp.fit(X_train, y_train_t); fit_secs = time.perf_counter() - t0
    mu_t, std_t = gp.predict(X_test, return_std=True)
    nlpd = float(np.mean(nlpds(y_test_t, mu_t, std_t)))

    mu_residual_space = inv(mu_t)
    if p in DELTA_TARGET: mu_residual_space = mu_residual_space + l1_te
    if DEMEAN_BY_SITE and site_mean_train is not None:
        mu = mu_residual_space + test_df.loc[test_df.index[keep_te] if p in DELTA_TARGET else test_df.index, "unique_id"]\
                                   .map(site_mean_train).fillna(global_mean_train).values
    else:
        mu = mu_residual_space

    rmse = float(np.sqrt(mean_squared_error(y_test_raw_sub, mu)))
    mae  = float(mean_absolute_error(y_test_raw_sub, mu))
    if p in DELTA_TARGET:
        y_pred_base = np.asarray(l1_te, dtype=float).ravel()
    else:
        y_pred_base = np.asarray(test_df[f"{p}_lag_1"].values, dtype=float).ravel()
    y_true_base = np.asarray(y_test_raw_sub, dtype=float).ravel()
    n = min(len(y_true_base), len(y_pred_base))
    mask_base = np.isfinite(y_true_base[:n]) & np.isfinite(y_pred_base[:n])
    if mask_base.sum() == 0:
        rmse_persist = np.nan; skill = np.nan
    else:
        rmse_persist = float(np.sqrt(mean_squared_error(y_true_base[:n][mask_base], y_pred_base[:n][mask_base])))
        skill = 1 - rmse / rmse_persist if rmse_persist > 0 else np.nan
    print(f"[{p}] fit={fit_secs:.1f}s | RMSE={rmse:.4f} | MAE={mae:.4f} | skill_vs_persist={skill:.3f} | NLPD={nlpd:.3f}")

    kernels_fitted[p] = gp.named_steps["gpr"].kernel_
    results.append([p, len(X_train), len(X_test), rmse, mae, rmse_persist, skill, nlpd])

print("\nValidation metrics:")
print(pd.DataFrame(results, columns=["Pollutant","n_train","n_test","RMSE","MAE","RMSE_persist","Skill","NLPD"]).to_string(index=False))

# ============================================================
# PM2.5 — Full fit on observed + site-level imputation with variance (2024)
# ============================================================
POLLUTANT = "PM2.5"
features_pm = ["x_km","y_km","t_week", f"{POLLUTANT}_lag_1", f"{POLLUTANT}_lag_2", f"{POLLUTANT}_lag_3"]
all_cols = ["unique_id","ds","County","Site Longitude","Site Latitude", POLLUTANT] + features_pm
dfa = df[all_cols].copy()

obs_mask  = dfa[POLLUTANT].notna()
feat_mask = dfa[features_pm].notna().all(axis=1)
train_all = dfa[obs_mask & feat_mask].copy()
missing   = dfa[~obs_mask & feat_mask].copy()

site_mean_all = train_all.groupby("unique_id")[POLLUTANT].mean()
global_mean_all = float(site_mean_all.mean())

X_train_all = train_all[features_pm].values
y_train_all = train_all[POLLUTANT].values
y_train_all_res = y_train_all - train_all["unique_id"].map(site_mean_all).fillna(global_mean_all).values

if PM_TF == "yeojohnson":
    pt_all = PowerTransformer(method="yeo-johnson", standardize=False)
    y_train_all_t = pt_all.fit_transform(y_train_all_res.reshape(-1,1)).ravel()
    inv_all = lambda z: pt_all.inverse_transform(np.asarray(z).reshape(-1,1)).ravel()
else:
    y_train_all_t = y_train_all_res.copy()
    inv_all = lambda z: np.asarray(z)

idx_space, idx_time, idx_lags = (0,1), (2,), (3,4,5)
kernel_pm = build_kernel_variogram_guided(POLLUTANT, idx_space, idx_time, idx_lags, sugg)
gp_all = make_gp(kernel_pm, restarts=RESTARTS_PER[POLLUTANT])

train_all_sub = thin_training(train_all, stride=STRIDE_ALL, max_total=MAX_TRAIN_ALL, recent_weeks=RECENT_WEEKS_ALL)
X_train_all_sub = train_all_sub[features_pm].values
y_train_all_sub = (train_all_sub[POLLUTANT].values
                   - train_all_sub["unique_id"].map(site_mean_all).fillna(global_mean_all).values)
if PM_TF == "yeojohnson":
    y_train_all_sub_t = pt_all.transform(y_train_all_sub.reshape(-1,1)).ravel()
else:
    y_train_all_sub_t = y_train_all_sub.copy()

t0 = time.perf_counter()
gp_all.fit(X_train_all_sub, y_train_all_sub_t)
print(f"\n[PM2.5] Full fit done in {time.perf_counter()-t0:.1f}s")
print("Fitted kernel:\n", gp_all.named_steps["gpr"].kernel_)

# Site-level predictions (missing rows) with variance
if len(missing):
    X_miss = missing[features_pm].values
    mu_t, std_t = gp_all.predict(X_miss, return_std=True)
    var_t = std_t**2
    if PM_TF == "yeojohnson":
        eps = 1e-4
        mu_res = inv_all(mu_t)
        up = inv_all(mu_t + eps); dn = inv_all(mu_t - eps)
        gprime = (up - dn) / (2*eps)
        var_res = (gprime**2) * var_t
    else:
        mu_res  = mu_t.copy()
        var_res = var_t.copy()
    site_shift = missing["unique_id"].map(site_mean_all).fillna(global_mean_all).values
    mu_all = mu_res + site_shift
    out_missing = missing.copy()
    out_missing[f"{POLLUTANT}_pred"] = mu_all
    out_missing[f"{POLLUTANT}_var"]  = var_res
    out_missing["is_imputed"] = True
else:
    out_missing = dfa.iloc[0:0].copy()

# Observed rows retain observed value; variance = 0
out_observed = dfa[obs_mask].copy()
out_observed[f"{POLLUTANT}_pred"] = out_observed[POLLUTANT].values
out_observed[f"{POLLUTANT}_var"]  = 0.0
out_observed["is_imputed"] = False

out_sites = pd.concat([out_observed, out_missing], ignore_index=True, sort=False)
out_sites = out_sites.sort_values(["unique_id","ds"]).reset_index(drop=True)

site_keep = ["ds","unique_id","County","Site Longitude","Site Latitude",
             POLLUTANT, f"{POLLUTANT}_pred", f"{POLLUTANT}_var","is_imputed"] + features_pm
out_sites = out_sites[site_keep]
out_sites.to_csv("pm25_2024_imputed_with_var.csv", index=False)
print(f"Wrote site-level file with variance: pm25_2024_imputed_with_var.csv")
print(f"Imputed rows: {int(out_sites['is_imputed'].sum())}")

# ============================================================
# County-week aggregation using GP joint covariance (MVE/GLS) — 2024
# ============================================================
from sklearn.gaussian_process.kernels import WhiteKernel

def _white_variance_from_kernel(k):
    """Sum WhiteKernel noise variances present in the fitted kernel."""
    if isinstance(k, WhiteKernel):
        return float(k.noise_level)**2
    total = 0.0
    for child in ("k1","k2"):
        if hasattr(k, child):
            total += _white_variance_from_kernel(getattr(k, child))
    return total

def county_week_from_gp(gp_pipeline, dfa, ds_week, county_name,
                        features, site_mean_all, global_mean_all,
                        transform="yeojohnson", inv_all=None,
                        include_obs_noise=True, ridge=1e-8):
    sub = dfa[(dfa["ds"] == ds_week) & (dfa["County"] == county_name)].copy()
    sub = sub[sub[features].notna().all(axis=1)]
    n = len(sub)
    if n == 0:
        return None
    X = sub[features].values
    mu_t, Sigma_t = gp_pipeline.predict(X, return_cov=True)
    if transform == "yeojohnson":
        if inv_all is None:
            raise ValueError("inv_all required when transform='yeojohnson'.")
        eps = 1e-4
        mu_res = inv_all(mu_t)
        up = inv_all(mu_t + eps); dn = inv_all(mu_t - eps)
        gprime = (up - dn) / (2*eps)
        J = gprime[:, None]
        Sigma_res = (J * Sigma_t) * gprime[None, :]
    else:
        mu_res = mu_t.copy()
        Sigma_res = Sigma_t.copy()
    if not include_obs_noise:
        sigma2_noise = _white_variance_from_kernel(gp_pipeline.named_steps["gpr"].kernel_)
        Sigma_res = Sigma_res.copy()
        np.fill_diagonal(Sigma_res, np.clip(np.diag(Sigma_res) - sigma2_noise, 0.0, np.inf))
    mu_sites = mu_res + sub["unique_id"].map(site_mean_all).fillna(global_mean_all).values
    ones = np.ones(n)
    Sigma_r = Sigma_res + ridge*np.eye(n)
    x = np.linalg.solve(Sigma_r, ones)
    denom = float(ones @ x)
    w = x / denom
    county_mean = float(w @ mu_sites)
    county_var  = float(1.0 / denom)
    return {"County": county_name, "ds": ds_week, "n_sites": n,
            "PM2.5_county_mean": county_mean, "PM2.5_county_var": max(county_var, 0.0)}

groups = (dfa[features_pm + ["County","ds"]]
          .dropna(subset=features_pm)
          .groupby(["County","ds"]).size().reset_index()[["County","ds"]])

rows = []
for _, row in groups.iterrows():
    res = county_week_from_gp(
        gp_pipeline=gp_all,
        dfa=dfa,
        ds_week=row["ds"],
        county_name=row["County"],
        features=features_pm,
        site_mean_all=site_mean_all,
        global_mean_all=global_mean_all,
        transform=PM_TF,
        inv_all=inv_all,
        include_obs_noise=INCLUDE_OBS_NOISE,
        ridge=RIDGE
    )
    if res is not None:
        rows.append(res)

county_df = pd.DataFrame(rows).sort_values(["County","ds"]).reset_index(drop=True)
county_df.to_csv("pm25_2024_by_county_week_cov.csv", index=False)
print(f"Wrote county-week file (MVE with covariance): pm25_2024_by_county_week_cov.csv")
print(county_df.head(10))



Non-NA counts (raw):
  PM2.5: 8351
  NO2: 4133
  CO: 1863

Computing variogram-based kernel hints…

=== Variogram-based suggestions ===
Pollutant      Sill  r95_time_weeks  r95_space_km  l_rbf_time  l_m32_time  l_rbf_space  l_m32_space                          bounds_rbf_time                         bounds_m32_time                        bounds_rbf_space                        bounds_m32_space  ACF_lag52  Suggest_Periodic
    PM2.5 21.116641       10.629394    142.520092    4.342522    2.240661    58.225014    30.043036   (2.171260922133147, 8.685043688532588) (1.1203306966564857, 4.481322786625943) (29.11250675457306, 116.45002701829225)  (15.02151797662548, 60.08607190650192)        NaN             False
      NO2 40.489036       11.689491    495.000000    4.775613    2.464128   202.226796   104.345307  (2.3878063757746593, 9.551225503098637)   (1.232064167499615, 4.92825666999846) (101.11339820677465, 404.4535928270986) (52.17265354870779, 208.69061419483117)        NaN             